# 音视频与嵌入内容

学习目标：能配置音视频控件、封面与字幕，并区分嵌入页面的来源、沙箱和功能权限。

前置知识：HTML 元素与属性、相对 URL、页面标题及浏览器开发者工具。

适用范围：WHATWG HTML Living Standard；使用现代浏览器观察，原生控件、解码支持和自动播放策略以实际浏览器为准。

环境准备：[环境配置与运行](README.md)。

工作目录：content/Web与应用开发/html；在浏览器运行配套页面。

本章媒体已提供：约四秒的方块画面与两声轻提示，没有人声。音视频为原创合成素材；本章直接使用这些文件，不需要 FFmpeg。

配套脚本：位于 scripts/08-media-and-embedded-content/。

1. [audio.html](scripts/08-media-and-embedded-content/audio.html)、[video.html](scripts/08-media-and-embedded-content/video.html)、[autoplay.html](scripts/08-media-and-embedded-content/autoplay.html)：音频、带字幕视频与静音自动播放。
2. [captions-zh.vtt](scripts/08-media-and-embedded-content/captions-zh.vtt)：两段中文定时文字，包含声音提示。
3. [iframe.html](scripts/08-media-and-embedded-content/iframe.html)、[frame.html](scripts/08-media-and-embedded-content/frame.html)：五种嵌入条件及被嵌入卡片；内置脚本只显示执行和 DOM 读取情况。
4. [objects.html](scripts/08-media-and-embedded-content/objects.html)：外部对象、嵌入内容与后备图像。
5. [signal.wav](scripts/08-media-and-embedded-content/signal.wav)、[signal.mp4](scripts/08-media-and-embedded-content/signal.mp4)、[signal.webm](scripts/08-media-and-embedded-content/signal.webm)、[poster.png](scripts/08-media-and-embedded-content/poster.png)：本地音视频与封面。
6. [serve.py](scripts/08-media-and-embedded-content/serve.py)：固定字幕、媒体响应类型的本机服务。

## 打开配套页面

Step 1：在两个已激活 Python 环境的终端中，分别从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/html
```

Step 2：在终端一启动主页面服务。

```bash
python scripts/08-media-and-embedded-content/serve.py --port 8008
```

Step 3：在终端二启动来源对照服务。

```bash
python scripts/08-media-and-embedded-content/serve.py --port 8108
```

Step 4：打开[视频入口](http://127.0.0.1:8008/video.html)和[嵌入对照](http://127.0.0.1:8008/iframe.html)。

两个服务都以 scripts/08-media-and-embedded-content/ 为根目录，因此浏览器地址从 /video.html 等文件名开始；8108 端口只为第 7 节跨源对照提供另一来源。

Notebook 的配套链接相对于 Notebook；HTML 中的 signal.mp4 等地址相对于该页面，不能再加一层 scripts 路径。字幕通过服务返回 text/vtt，不使用 file: 地址做本章实验。

端口被占用时先检查自己已有的本章服务。

Step 5：在两个服务终端分别按 Ctrl+C 停止服务。

中断会显示原始 KeyboardInterrupt；服务的 with 上下文随后关闭监听端口。

## 1 标签总览

| 标签 | 中文名称／含义 | 用途 |
| --- | --- | --- |
| &lt;audio&gt; | 音频元素 | 播放声音 |
| &lt;video&gt; | 视频元素 | 播放画面及可选声音 |
| &lt;source&gt; | 资源候选 | 提供同一媒体的不同格式 |
| &lt;track&gt; | 定时文字轨道 | 关联字幕等文字资源 |
| &lt;iframe&gt; | 内联框架 | 嵌入另一个 HTML 页面 |
| &lt;object&gt; | 外部对象 | 嵌入资源并容纳后备内容 |
| &lt;embed&gt; | 外部内容嵌入 | 嵌入浏览器支持的资源类型 |

&lt;source&gt;、&lt;track&gt;、&lt;embed&gt; 是空元素；本章其他标签保留开始和结束标签。

## 2 &lt;audio&gt;：手动播放声音

单个音频文件直接写在 src 中；controls 请求原生播放、暂停等控件。aria-label 为本例播放器提供名称。

```html
<audio src="signal.wav" controls preload="metadata" aria-label="两次轻提示音">
  此浏览器不支持 audio，请使用下面的文件入口。
</audio>
<!-- 预期：打开页面不发声；主动按播放后时间推进并在约四秒处结束。 -->
<!-- 用 Tab 找到控件，再用浏览器支持的键盘操作播放和暂停。 -->
```

配套文件：[scripts/08-media-and-embedded-content/audio.html](scripts/08-media-and-embedded-content/audio.html) · [浏览器预览](http://127.0.0.1:8008/audio.html)

preload 是预加载提示，不保证请求次数或下载量：

- none：建议不预加载。
- metadata：建议获取时长等元数据。
- auto：允许预加载整个媒体。

播放器内部文字面向不支持该元素的浏览器；支持元素但加载失败时，不保证显示这些文字。把文字说明与下载入口放在播放器外，才能一直访问。

```html
<p>文字说明：约 0.25 秒、2.25 秒各有一声短音，其余时间安静。</p>
<p><a href="signal.wav" download>下载四秒 WAV 音频</a></p>
```

配套文件：[scripts/08-media-and-embedded-content/audio.html](scripts/08-media-and-embedded-content/audio.html) · [浏览器预览](http://127.0.0.1:8008/audio.html)

## 3 &lt;video&gt;：封面与格式候选

视频可以包含声音。下面保留控件、默认静音；播放器外另有完整文字说明和下载链接。

- poster：播放前可显示的图片地址，不是视频资源，也不代替文字说明。
- width、height：以不带单位的整数指定显示尺寸，单位为 CSS 像素，不会重新编码视频。
- playsinline：提示在页面内播放，缺失它不等于一定全屏。
- muted：默认静音；用户可主动取消静音。

```html
<video id="demo-video" controls muted playsinline preload="metadata"
  poster="poster.png" width="320" height="180"
  aria-label="方块换位置，约四秒">
  <source src="signal.webm" type="video/webm">
  <source src="signal.mp4" type="video/mp4">
  <track kind="captions" src="captions-zh.vtt" srclang="zh-CN"
    label="中文（含声音提示）" default>
  此浏览器不支持 video，请使用下方文字说明或下载链接。
</video>
<!-- 预期：点击播放前显示封面；播放后方块在两秒附近切换位置。 -->
<!-- 在 Network 或媒体元素 currentSrc 中看实际选中的资源，不能假定两个都播放。 -->
<!-- 约四秒结束；容器时长可能略多于四秒，封面本身不能证明视频解码成功。 -->
```

配套文件：[scripts/08-media-and-embedded-content/video.html](scripts/08-media-and-embedded-content/video.html) · [浏览器预览](http://127.0.0.1:8008/video.html)

&lt;source&gt; 列出同一内容的格式候选，按文档顺序供浏览器选择，不是播放列表。使用内部候选时不再给 &lt;video&gt; 写 src；&lt;source&gt; 在这里用 src，在 &lt;picture&gt; 中则用 srcset。

type 是 MIME 媒体类型提示，不负责转换格式。MP4、WebM 是容器，H.264、VP8 等是编码格式；支持某个容器不代表支持其中所有编码。本章 MP4 使用 H.264/AAC，WebM 使用 VP8/Vorbis。

&lt;track&gt; 位于 &lt;source&gt; 之后，作用见下一节。

## 4 &lt;track&gt;：关联字幕

定时文字随播放时间呈现。下面是视频内部的轨道，字幕文件已提供，无需先生成。

```html
<track kind="captions" src="captions-zh.vtt" srclang="zh-CN"
  label="中文（含声音提示）" default>
```

配套文件：[scripts/08-media-and-embedded-content/video.html](scripts/08-media-and-embedded-content/video.html) · [浏览器预览](http://127.0.0.1:8008/video.html)

- src：WebVTT 文件地址，本例与页面同源。跨源字幕还需要父媒体元素的 crossorigin 配置及服务器许可，不能只改地址。
- kind：轨道用途，本例 captions 包括两次短音信息。
- srclang：轨道语言，zh-CN 表示本例中文。kind="subtitles" 时必须提供。
- label：字幕菜单中的名称。
- default：建议优先启用，仍可能被用户偏好覆盖；本例仅设一个。

kind 的其他常见值：

- subtitles：对白转写或翻译等字幕。
- descriptions：对视觉内容的文字描述。
- chapters：用于导航的章节标题。
- metadata：供脚本读取的数据。

&lt;track&gt; 可关联 &lt;audio&gt; 或 &lt;video&gt;，但原生音频控件不保证像视频一样显示字幕。本例音频提供常显文字，视频提供字幕与完整说明。

### 4.1 WebVTT：时间段与文字

WebVTT 是 UTF-8 文本格式，响应类型为 text/vtt。文件以 WEBVTT 开头，空一行后写字幕条目；空行分隔条目。

时间写成“小时:分钟:秒.毫秒”，例如 00:00:02.000 是媒体起点之后两秒。--> 分隔开始与结束时间；结束必须晚于开始，后续条目的开始时间不能早于前一条的开始时间。

```text
WEBVTT

00:00:00.000 --> 00:00:02.000
[一声短提示音] 黄色方块在左侧。

00:00:02.000 --> 00:00:04.000
[一声短提示音] 黄色方块切到右侧。
```

配套文件：[scripts/08-media-and-embedded-content/captions-zh.vtt](scripts/08-media-and-embedded-content/captions-zh.vtt) · [浏览器预览](http://127.0.0.1:8008/captions-zh.vtt)

方括号描述声音，文字与文件中的两声提示相对应，不是烧录进视频画面的字幕。到[视频页面](http://127.0.0.1:8008/video.html)选择中文轨道，观察两秒前后的文字切换；不要仅凭 default 推断已经启用。

## 5 autoplay：受策略约束的自动播放

controls、muted、autoplay 是布尔属性，存在即表示启用。关闭自动播放要删除 autoplay，不能写 autoplay="false"。

浏览器通常限制未经用户交互的有声自动播放。静音视频更容易被允许，但仍受站点设置、用户偏好和浏览器策略影响；用脚本调用 play() 也受策略约束。

```html
<video controls autoplay muted playsinline preload="none"
  src="signal.mp4" poster="poster.png" width="320" height="180"
  aria-label="静音自动播放的方块视频">
  <track kind="captions" src="captions-zh.vtt" srclang="zh-CN"
    label="中文（含声音提示）" default>
  此浏览器不支持 video。
</video>
<!-- 观察而不预设策略：刷新后不点击，查看时间是否推进和画面是否切换。 -->
<!-- 若时间保持为零，手动播放应仍可使用；不能据此直接认定文件损坏。 -->
<!-- 操作：保持 muted；观察后暂停或关闭这个标签页。 -->
```

配套文件：[scripts/08-media-and-embedded-content/autoplay.html](scripts/08-media-and-embedded-content/autoplay.html) · [浏览器预览](http://127.0.0.1:8008/autoplay.html)

自动播放可能优先于 preload="none"。不要只看资源已下载、写了 autoplay 或处于静音，就认定播放已发生；观察时间是否推进，并保留手动播放和暂停入口。

## 6 &lt;iframe&gt;：嵌入一个有名称的页面

&lt;iframe&gt; 创建独立的嵌入文档环境，不是把子页面源码复制进父页面。配套页面先用 A 卡片观察基本写法。

- src：子页面地址；frame.html 从父页面所在目录解析。
- title：简述框架内容，帮助辅助技术识别；与子页面 &lt;head&gt; 中的 &lt;title&gt; 各有用途。
- loading：eager 是默认加载方式，lazy 提示接近视口时再加载。具体距离由浏览器决定，且延迟加载需启用 JavaScript。
- allow：功能权限策略；本例禁用全屏、摄像头、麦克风，第 8 节解释。

```html
<iframe id="same" src="frame.html" title="A：同源普通卡片"
  width="320" height="250" loading="eager"
  allow="fullscreen 'none'; camera 'none'; microphone 'none'"></iframe>
```

配套文件：[scripts/08-media-and-embedded-content/iframe.html](scripts/08-media-and-embedded-content/iframe.html) · [浏览器预览](http://127.0.0.1:8008/iframe.html)

被嵌入的 [scripts/08-media-and-embedded-content/frame.html](scripts/08-media-and-embedded-content/frame.html) 用脚本显示“脚本已执行”，并尝试读取父页面标题。脚本失败时保留或更新状态文字，供下面比较来源与限制。

## 7 sandbox：分开判断脚本和来源

普通 HTTP 页面是否同源，比较 URL 的方案、主机和端口。本例父页面的来源为 [8008 端口](http://127.0.0.1:8008/iframe.html)，[8108 端口的卡片](http://127.0.0.1:8108/frame.html) 因端口不同而跨源，即使两台服务读取同一个文件。

能显示跨源页面，不等于能直接读取它的 DOM（浏览器建立的文档元素树）。sandbox 再为嵌入文档施加一组限制：

- sandbox=""：启用全部沙箱限制，与省略整个属性不同。
- allow-scripts：解除脚本执行限制，不恢复文档来源。
- allow-same-origin：保留文档原本的来源，不会把跨源文档改成同源。

不包含 allow-same-origin 时，文档使用不透明来源（opaque origin），不能通过通常的同源比较。许可关键字用空格分隔；未解除的限制仍保留。

### 7.1 B、C：同一地址，只改变脚本许可

B 卡片可以显示，但禁止脚本且不保留原来源。

```html
<iframe id="locked" src="frame.html" title="B：启用全部沙箱限制的卡片"
  width="320" height="250" loading="lazy" sandbox=""
  allow="fullscreen 'none'; camera 'none'; microphone 'none'"></iframe>
```

配套文件：[scripts/08-media-and-embedded-content/iframe.html](scripts/08-media-and-embedded-content/iframe.html) · [浏览器预览](http://127.0.0.1:8008/iframe.html)

C 只解除脚本限制。卡片能执行脚本，但读取父页面仍遭遇 SecurityError。

```html
<iframe id="scripts" src="frame.html" title="C：仅允许脚本的卡片"
  width="320" height="250" loading="lazy" sandbox="allow-scripts"
  allow="fullscreen 'none'; camera 'none'; microphone 'none'"></iframe>
<!-- 预期：脚本执行，但卡片读取父页面遭遇 SecurityError。 -->
```

配套文件：[scripts/08-media-and-embedded-content/iframe.html](scripts/08-media-and-embedded-content/iframe.html) · [浏览器预览](http://127.0.0.1:8008/iframe.html)

### 7.2 D、E：保留来源，不会创造同源

D 保留原本的 8008 来源，却没有解除脚本限制。因此子页面脚本不执行，同源父页面仍能读取它。

```html
<iframe id="origin" src="frame.html" title="D：保留来源但不执行脚本的卡片"
  width="320" height="250" loading="lazy" sandbox="allow-same-origin"
  allow="fullscreen 'none'; camera 'none'; microphone 'none'"></iframe>
```

配套文件：[scripts/08-media-and-embedded-content/iframe.html](scripts/08-media-and-embedded-content/iframe.html) · [浏览器预览](http://127.0.0.1:8008/iframe.html)

E 的脚本可以执行，保留的却是 8108 来源，仍与父页面跨源。

```html
<iframe id="cross" src="http://127.0.0.1:8108/frame.html"
  title="E：8108 端口的跨源卡片" width="320" height="250"
  loading="lazy" sandbox="allow-scripts allow-same-origin"
  allow="fullscreen 'none'; camera 'none'; microphone 'none'"></iframe>
```

配套文件：[scripts/08-media-and-embedded-content/iframe.html](scripts/08-media-and-embedded-content/iframe.html) · [浏览器预览](http://127.0.0.1:8008/iframe.html)

不要把 E 改成同源不可信页面后继续同时开启 allow-scripts 与 allow-same-origin：子页面可能移除 sandbox 并重新加载，从而脱离沙箱。sandbox 不负责清洗内容，也不保证禁止所有网络活动，仍需控制嵌入来源和必要能力。

### 7.3 对照父子页面的读取结果

先滚动到五个框架，等卡片加载后再点“读取各框架标题”。

```html
<button id="inspect" type="button">读取各框架标题</button>
<pre id="results" aria-live="polite">尚未检查</pre>
```

配套文件：[scripts/08-media-and-embedded-content/iframe.html](scripts/08-media-and-embedded-content/iframe.html) · [浏览器预览](http://127.0.0.1:8008/iframe.html)

- A：子页面脚本执行，父子页面可相互读取标题。
- B：子页面脚本不执行，父页面读取受阻。
- C：子页面脚本执行，父子页面 DOM 读取均受阻。
- D：子页面脚本不执行，父页面可读取子页面标题。
- E：子页面脚本执行，父子页面 DOM 读取均受阻。

观察器使用 contentWindow、parent.document 等浏览器 API；完整 JavaScript 留在配套文件。Console 中 B、D 的脚本拒绝属于预期，不能当成资源 404；其他加载错误仍需排查。

这里的异常捕获专门用于同源限制对照：把预期 SecurityError 与可读取的标题并列展示；其他异常继续抛出，不把这种写法套到普通页面。

## 8 allow：功能权限与嵌入来源边界

allow 设置框架的 Permissions Policy（权限策略），与 sandbox 许可关键字不同。下面是 A 框架的完整写法，其他框架采用相同功能限制。

```html
<iframe id="same" src="frame.html" title="A：同源普通卡片"
  width="320" height="250" loading="eager"
  allow="fullscreen 'none'; camera 'none'; microphone 'none'"></iframe>
```

配套文件：[scripts/08-media-and-embedded-content/iframe.html](scripts/08-media-and-embedded-content/iframe.html) · [浏览器预览](http://127.0.0.1:8008/iframe.html)

- fullscreen 'none'：禁止全屏。
- camera 'none'：禁止摄像头。
- microphone 'none'：禁止麦克风。

单引号属于属性值，多条指令用分号分隔。这些策略不能覆盖上级 Permissions-Policy 响应头的限制；允许某项功能也不等于已经取得用户授权。功能支持有浏览器差异，本章只读取 fullscreenEnabled，不请求设备或全屏。

资源提供方还可用 Content-Security-Policy 的 frame-ancestors 或 X-Frame-Options 限制被哪些页面嵌入。无法显示时同时检查 Network、Console 和响应头，不能通过放宽 sandbox 绕过提供方的限制。

## 9 &lt;object&gt; 与 &lt;embed&gt;（补充）

两者可嵌入浏览器支持的外部资源，处理效果取决于资源类型和浏览器。音视频通常用 &lt;audio&gt;、&lt;video&gt;，HTML 页面用 &lt;iframe&gt;；普通图片用 &lt;img&gt;。

- &lt;object&gt; 用 data 指定资源，可以包含后备内容。
- &lt;embed&gt; 用 src 指定资源，是空元素，没有内部后备内容。

```html
<object data="poster.png" type="image/png" width="320" height="180"
  aria-label="黄色方块位于深蓝画面左侧">
  <img src="poster.png" width="320" height="180"
    alt="黄色方块位于深蓝画面左侧">
</object>
```

配套文件：[scripts/08-media-and-embedded-content/objects.html](scripts/08-media-and-embedded-content/objects.html) · [浏览器预览](http://127.0.0.1:8008/objects.html)

```html
<embed src="poster.png" type="image/png" width="320" height="180"
  title="黄色方块位于深蓝画面左侧">
<p><a href="poster.png">单独打开封面图像</a></p>
```

配套文件：[scripts/08-media-and-embedded-content/objects.html](scripts/08-media-and-embedded-content/objects.html) · [浏览器预览](http://127.0.0.1:8008/objects.html)

本例使用同一张本地 PNG，&lt;object&gt; 用 aria-label 提供名称，&lt;embed&gt; 用 title 描述内容。若原图不可用，&lt;object&gt; 内的同图后备资源也可能失败；练习只改变 data，保留真实可用的后备地址。这个例子不能证明浏览器支持 PDF 等其他嵌入资源。

## 本章小结

- &lt;audio&gt;、&lt;video&gt; 提供播放入口；格式候选不等于播放列表。
- preload 是提示，poster 是封面，autoplay 受策略约束。始终保留控件与文字入口。
- &lt;track&gt; 关联定时文字，字幕的语言、时间、声音信息和来源应对应实际内容。
- 嵌入是否显示、脚本是否执行、DOM 是否可读、功能是否可用要分别判断。allow-same-origin 保留原来源，不创造同源关系。

## 练习

在 scripts/08-media-and-embedded-content/ 中复制示例后修改：

（1）将 video.html 复制为 practice-video.html，把 MP4 候选移到 WebM 前面。检查：仍可手动播放，记录实际 currentSrc 或 Network 请求，不能只根据候选列表认定结果。

（2）将 captions-zh.vtt 复制为 practice-captions.vtt，让视频副本的 &lt;track&gt; 指向它。把第二条拆成 2～3 秒、3～4 秒两条；第一条保留换位置及短音说明，后一条只说明方块停在右侧。检查：三个时间段显示对应文字，不给最后一秒编造声音。

（3）将 iframe.html 复制为 practice-iframe.html，删除 C 的 allow-scripts 值，保留空 sandbox。检查：刷新后 C 的脚本不执行，父页面仍无法读取它；A、D、E 的条件不变。

（4）将 objects.html 复制为 practice-objects.html，只把 &lt;object&gt; 的 data 改为 missing-poster.png。检查：Network 显示该资源失败，内部后备图像可用，&lt;embed&gt; 不受影响。

### 提示

副本通过原页面 URL 替换文件名访问。第二题用空行分隔字幕条目；第三题不要删除 sandbox 属性本身，也不要对同源不可信页面同时放开脚本和来源。用后可删除自行创建的副本。

### 参考解析

1. MP4 候选移到前面后，浏览器先考虑它；仍以实际 currentSrc、播放时间和画面核对成功，不能仅凭顺序声称解码通过。
2. 保留 0～2 秒条目，再设 2～3 秒“[一声短提示音] 黄色方块切到右侧。”和 3～4 秒“黄色方块停在右侧。”。空行分隔三条，轨道 src 指向副本；时间段变化不改变音视频文件。
3. C 变为空 sandbox 后与 B 的限制一致：卡片脚本不执行，父页面仍无法读取其 DOM。保留 sandbox 属性，不能把删除许可值误做成删除全部限制。
4. 只把 object 的 data 改为不存在的文件时，浏览器显示内部仍可加载的 poster.png。embed 继续使用原 src；这是 object 后备内容的对照，不是额外编写脚本兜底。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| WHATWG HTML | [&lt;audio&gt;](https://html.spec.whatwg.org/multipage/media.html#the-audio-element)、[&lt;video&gt;](https://html.spec.whatwg.org/multipage/media.html#the-video-element)、[&lt;track&gt;](https://html.spec.whatwg.org/multipage/media.html#the-track-element)与[媒体元素](https://html.spec.whatwg.org/multipage/media.html#media-elements)的内容、资源、控件和轨道；[&lt;source&gt;](https://html.spec.whatwg.org/multipage/embedded-content.html#the-source-element)的媒体与图像属性区别；[&lt;iframe&gt;](https://html.spec.whatwg.org/multipage/iframe-embed-object.html#the-iframe-element)的 sandbox、真实来源及同源双许可风险；同页的 [&lt;object&gt;](https://html.spec.whatwg.org/multipage/iframe-embed-object.html#the-object-element)、[&lt;embed&gt;](https://html.spec.whatwg.org/multipage/iframe-embed-object.html#the-embed-element)与后备内容。 |
| W3C | [Captions/Subtitles](https://www.w3.org/WAI/media/av/captions/)的声音信息与字幕用途；[H64：框架 title](https://www.w3.org/WAI/WCAG22/Techniques/html/H64)的名称和页面标题区别；[WebVTT §4.1 文件结构](https://www.w3.org/TR/webvtt1/#webvtt-file-structure)与 §10.1 text/vtt，核对字幕文本和时间段。 |
| MDN | [&lt;audio&gt;](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/audio#attributes)、[&lt;video&gt;](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/video#attributes)的属性和后备文字；[&lt;track&gt;](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/track#attributes)的 kind、srclang、default 和同源条件；[WebVTT](https://developer.mozilla.org/en-US/docs/Web/API/WebVTT_API/Web_Video_Text_Tracks_Format#overview)的编码、响应类型和 Cue timings；[自动播放指南](https://developer.mozilla.org/en-US/docs/Web/Media/Guides/Autoplay#autoplay_availability)的策略和 play()；[&lt;iframe&gt;](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/iframe#attributes)、[同源策略](https://developer.mozilla.org/en-US/docs/Web/Security/Defenses/Same-origin_policy#definition_of_an_origin)、[权限策略](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Permissions_Policy#embedded_frame_syntax)的来源、loading、sandbox 和 allow；[frame-ancestors](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/Content-Security-Policy/frame-ancestors)、[X-Frame-Options](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/X-Frame-Options)的嵌入限制；[contentWindow](https://developer.mozilla.org/en-US/docs/Web/API/HTMLIFrameElement/contentWindow#description)、[fullscreenEnabled](https://developer.mozilla.org/en-US/docs/Web/API/Document/fullscreenEnabled#value)的观察边界；[媒体容器](https://developer.mozilla.org/en-US/docs/Web/Media/Guides/Formats/Containers)的容器与编码格式区别。 |
| Python 3.12 | [http.server](https://docs.python.org/3.12/library/http.server.html#http.server.SimpleHTTPRequestHandler)的 directory、extensions_map 和 ThreadingHTTPServer，支持本章固定目录与媒体类型预览。 |